# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya - Exploration with `mlcroissant`

This notebook provides a step-by-step guide for loading and exploring the [FAIR^2](https://sen.science/doi/10.71728/senscience.y7m0-f273) dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

In [ ]:
# Ensure `mlcroissant` is installed
!pip install --quiet mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json"

# Load the dataset via mlcroissant
dataset = mlc.Dataset(croissant_url)

# Display high-level metadata
print(f"Dataset Title: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}\n")
print(f"Identifier: {dataset.metadata.identifier}")
print(f"Data Biases: {getattr(dataset.metadata, 'dataBiases', None)}\n")

## 2. Data Overview

Let's review available record sets (`@id`s), their fields, and the dataset structure.

> All entities are referenced by their `@id` (unique identifier within the schema).

In [ ]:
# List available record sets and their @id, name, fields and field @ids

if not dataset.metadata.recordSet:
    print("No record sets defined in the top-level metadata.")
else:
    for record_set in dataset.metadata.recordSet:
        print(f"\nRecordSet @id: {record_set['@id']}")
        print(f"  Name: {record_set.get('name', '<no name>')}")
        if 'field' in record_set:
            print("  Fields:")
            for field in record_set['field']:
                print(f"    - Field @id: {field['@id']} | name: {field.get('name', '<no name>')}")
        else:
            print("  <No fields listed>")
# If recordSet is empty (as in this metadata), let's try to access found record sets via the mlcroissant API:

record_set_descriptions = []
try:
    for rs in dataset.record_sets:
        rs_id = getattr(rs, '@id', None)
        name = getattr(rs, 'name', None)
        print(f"Found RecordSet @id: {rs_id} | name: {name}")
        record_set_descriptions.append((rs_id, name))
except AttributeError:
    print("No record sets found via API.")

## 3. Data Extraction

We'll extract data records using available record set `@id`s. This step stores your tabular data as pandas DataFrames for analysis.

### Referencing by `@id`

All data extraction refers to record sets, fields, or columns by their Croissant `@id`.

In [ ]:
# Determine all record set @id values from the dataset (fallback by inspecting dataset)
record_set_ids = []
if dataset.metadata.recordSet:
    # If metadata.recordSet is a list of dicts with @id
    record_set_ids = [rs['@id'] for rs in dataset.metadata.recordSet if '@id' in rs]
else:
    # Try to infer from dataset API
    try:
        for rs in dataset.record_sets:
            if hasattr(rs, '@id'):
                record_set_ids.append(getattr(rs, '@id'))
    except Exception as e:
        print("Could not auto-extract record sets:", str(e))

if not record_set_ids:
    # Try to display possible options manually
    print("No record sets found. Please refer to the Croissant schema for @id values.")
else:
    print("Record set @ids found:")
    for ridx, rid in enumerate(record_set_ids):
        print(f"  [{ridx}] {rid}")

# For demonstration, we'll attempt to extract from the first available record set (if present)
dataframes = {}
if record_set_ids:
    for rs_id in record_set_ids:
        try:
            records_iter = dataset.records(record_set=rs_id)
            data = list(records_iter)
            if data:
                df = pd.DataFrame(data)
                dataframes[rs_id] = df
                print(f"Loaded DataFrame for record set @id: {rs_id} (shape: {df.shape})")
        except Exception as e:
            print(f"Failed to load records for {rs_id}: {e}")

    # List columns for the first DataFrame
    if dataframes:
        first_rs_id = list(dataframes.keys())[0]
        print(f"\nFirst DataFrame columns for record set @id '{first_rs_id}':")
        print(dataframes[first_rs_id].columns.tolist())
        dataframes[first_rs_id].head()

## 4. Exploratory Data Analysis (EDA)

In this section, we demonstrate data processing by referencing all columns and fields by their `@id`. We'll show how to:
- Filter records based on numeric field thresholds
- Normalize a numeric column
- Group records by a categorical field

> **Ensure you use the `@id` of the fields as the column keys in DataFrames.**

In [ ]:
# Example usage: select a numeric field and a group field by their `@id`

# Replace these with actual @id values present in the DataFrame columns.
# Inspect DataFrame columns to pick candidate fields:
if dataframes:
    first_rs_id = list(dataframes.keys())[0]
    df = dataframes[first_rs_id]
    print(f"Available columns (@id): {df.columns.tolist()}")
    
    # As an example, try to detect a numeric column and a group field
    # We'll pick the first column that looks numeric
    numeric_field_id = None
    group_field_id = None
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
            break
    for col in df.columns:
        if pd.api.types.is_string_dtype(df[col]) or pd.api.types.is_categorical_dtype(df[col]):
            group_field_id = col
            break
    if numeric_field_id is not None:
        print(f"Using numeric field: {numeric_field_id}")
        threshold = df[numeric_field_id].mean() if pd.api.types.is_numeric_dtype(df[numeric_field_id]) else 0
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records where {numeric_field_id} > {threshold} (rows: {len(filtered_df)})")

        # Normalize
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Grouping, if group_field available
        if group_field_id is not None and group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
            print(f"\nGrouped mean of {numeric_field_id} by {group_field_id}:")
            print(grouped_df.head())
    else:
        print("Could not find any numeric field in columns.")

## 5. Visualization

Let's visualize data distributions with matplotlib and seaborn, referencing columns by their `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and numeric_field_id is not None:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    # If we have a group field, plot group means
    if group_field_id is not None:
        group_means = df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        plt.figure(figsize=(10, 5))
        sns.barplot(data=group_means, x=group_field_id, y=numeric_field_id)
        plt.title(f"Mean of {numeric_field_id} by {group_field_id}")
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.xlabel(group_field_id)
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()

## 6. Conclusion

In this notebook, we demonstrated loading and exploring a Croissant-defined dataset using the `mlcroissant` library. All queries and processing referenced entities by their Croissant `@id` values for full reproducibility.

- The FAIR^2 dataset provides rich microdata for understanding the adoption of indigenous and modern knowledge in rangeland management in Northern Kenya.
- We showed how to enumerate available record sets/fields, extract tabular data, filter and normalize values, and perform grouped analysis and visualization—all driven by the schema's semantic identifiers (`@id`).

You can further expand the analysis by leveraging additional fields and combinations of record sets as described in the schema.